# ТАСКА 1

Обучить автоэнкодер с квантизованным бутылочным горлом

## Либы

In [ ]:
!pip install --upgrade pip ninja setuptools wheel
!pip install "xarray[io]" zarr gcsfs fsspec cartopy shapely metpy pvlib xbatcher xarray_regrid torchac

## Предобработка данных

In [139]:
import xarray as xr
import xarray_regrid
import numpy as np

def get_dataset025():
    "Датасет с сеткой 0.25 градусов"
    rename_dict_ground = {
        "2m_temperature": "2t",
        "mean_sea_level_pressure": "msl",
        "10m_u_component_of_wind": "10u",
        "10m_v_component_of_wind": "10v",
        "total_precipitation_6hr": "tp",
        "sea_surface_temperature": "sst",
        "total_column_water_vapour": "tcwv",
        "total_cloud_cover": "tcc",
    }
    rename_dict_air = {
        "temperature": "t",
        "u_component_of_wind": "u",
        "v_component_of_wind": "v",
        "geopotential": "z",
        "specific_humidity": "q",
    }
    
    
    url = "gs://weatherbench2/datasets/era5/1959-2023_01_10-wb13-6h-1440x721_with_derived_variables.zarr"
    
    ds = xr.open_dataset(
      url,
      engine="zarr",
      consolidated=True,
      storage_options={"token": "anon"},
      chunks="auto",
    )
    
    # наземные показатели
    near_ground = [
      "2m_temperature",
      "mean_sea_level_pressure",
      "10m_u_component_of_wind",
      "10m_v_component_of_wind",
      "total_precipitation_6hr",
      "sea_surface_temperature",
      "total_column_water_vapour",
      "total_cloud_cover",
    ]
    
    isobar_types = [1000, 925, 850, 700] #изобарные значения
    
    air_types = [ #воздушные показатели
      "temperature",
      "u_component_of_wind",
      "v_component_of_wind",
      "geopotential",
      "specific_humidity",
    ]
    
    # Общий временной срез для обоих типов данных
    START_TIME = '2014-01-01'
    END_TIME = '2019-12-31'
    time_slice = slice(START_TIME, END_TIME)
    
    # выделяем приземные переменные
    ds_ground = ds[near_ground].sel(time=time_slice).rename(rename_dict_ground)
    
    # выделяем атмосферные переменные
    ds_air = ds[air_types].sel(level=isobar_types, time=time_slice)
    
    air_vars = {}
    for air in air_types:
        for bar in isobar_types:
            da = ds_air[air].sel(level=bar).drop_vars("level") # срез по уровням
            da.name = f"{rename_dict_air[air]}_{bar}"
            air_vars[da.name] = da
    
    # объединение
    ds_ground_dict = {name: ds_ground[name] for name in ds_ground.data_vars}
    all_vars = {**ds_ground_dict, **air_vars}
    ds = xr.merge(list(all_vars.values()))
    
    return ds

def get_dataset05():
    "Датасет с сеткой 0.5 градусов"
    ds = get_dataset025().to_dataset(dim="variable")
    
    target_grid = xr.Dataset(
        {
          "latitude": (["latitude"], np.arange(-89.75, 90.0, 0.5)),
          "longitude": (["longitude"], np.arange(0.0, 360.0, 0.5)),
        }
    )
    
    ds = ds.regrid.conservative(target_grid)
    return ds

def compute_dataset_stats(ds, sample_fraction=0.02):
    """
    Быстро считает среднее и std для каждой переменной по случайной временной подвыборке.
    Учитывает веса широт, чтобы избежать искажений на полюсах.
    """
    
    # Случайная подвыборка по time
    total_times = len(ds.time)
    n_samples = max(1, int(total_times * sample_fraction))
    random_indices = np.random.choice(total_times, size=n_samples, replace=False)
    random_indices.sort()
    
    ds_sampled = ds.isel(time=random_indices)
    
    weights = np.cos(np.deg2rad(ds_sampled.latitude))
    weights /= weights.mean()
    
    means = {}
    stds = {}
    
    for var_name in ds_sampled.data_vars:
        weighted_var = ds_sampled[var_name].weighted(weights)
        
        v_mean = float(weighted_var.mean().values)
        
        v_variance = float(((ds_sampled[var_name] - v_mean) ** 2).weighted(weights).mean().values)
        v_std = np.sqrt(v_variance)
        
        if v_std == 0: 
            v_std = 1.0
            
        means[var_name] = v_mean
        stds[var_name] = v_std
        print(f"Переменная {var_name:8} | Среднее: {v_mean:12.4f} | Std: {v_std:12.4f}")
        
    return means, stds


In [140]:
import torch

class WeatherAutoencoderLoader:
    def __init__(self, batch_generator, means_dict, stds_dict, device="cuda"):
        self.bgen = batch_generator
        self.var_names = None
        self.device = torch.device(device)
        self.means_dict = means_dict
        self.stds_dict = stds_dict

    def __iter__(self):
        for batch in self.bgen:
            if self.var_names is None:
                self.var_names = list(batch.data_vars)
                self.means_tensor = torch.tensor([self.means_dict[name] for name in self.var_names], dtype=torch.float32).view(-1, 1, 1, 1)
                self.stds_tensor = torch.tensor([self.stds_dict[name] for name in self.var_names], dtype=torch.float32).view(-1, 1, 1, 1)
                
                if self.device.type == "cuda":
                    self.means_tensor = self.means_tensor.to(self.device)
                    self.stds_tensor = self.stds_tensor.to(self.device)

            target_dims = ('time', 'latitude', 'longitude')
            
            data_arrays = [batch[var].transpose(*target_dims).values for var in self.var_names]
            data = np.stack(data_arrays, axis=0) 

            # ЗАЩИТА: Пропускаем неполные батчи с краев карты Земли
            if data.shape[2] != 128 or data.shape[3] != 128:
                continue

            tensor_batch = torch.as_tensor(data, dtype=torch.float32)
            if self.device.type == "cuda":
                tensor_batch = tensor_batch.pin_memory().to(self.device, non_blocking=True)

            tensor_batch = torch.nan_to_num(tensor_batch, nan=0.0, posinf=1.0, neginf=-1.0)
            
            # Z-score нормализация (вход и таргет масштабированы одинаково)
            tensor_batch = (tensor_batch - self.means_tensor) / self.stds_tensor

            # Time, Channels, Lat, Lon
            tensor_batch = tensor_batch.permute(1, 0, 2, 3)

            yield tensor_batch, tensor_batch


            
    def __len__(self):
        return len(self.bgen)

In [141]:
import xbatcher as xb

device = "cuda" if torch.cuda.is_available() else "cpu"
print("работаем на", device)

import dask
dask.config.set(scheduler='threads', num_workers=4) # Настраиваем dask на параллельную фоновую загрузку через потоки (threads)

print("Загрузка датасета 0.25*")
ds025 = get_dataset025()

print("Создание батчгенератора")
bgen = xb.BatchGenerator(
    ds025,
    input_dims={'time': 8, 'latitude': 128, 'longitude': 128},
    input_overlap={'time': 0, 'latitude': 0, 'longitude': 0},
    preload_batch=True,
)

print("Подсчет статистических данных")
# means_dict, stds_dict = compute_dataset_stats(ds025, sample_fraction=0.005)
# было вычислено строчкой выше и захардкожено для быстроты работы

работаем на cuda
Загрузка датасета 0.25*
Создание батчгенератора
Подсчет статистических данных


In [142]:
means_dict = {
    '2t': 287.54571533203125,
    'msl': 101144.828125,
    '10u': -0.37171676754951477,
    '10v': 0.14705954492092133,
    'tp': 0.0007431305712088943,
    'sst': 291.5942687988281,
    'tcwv': 24.714500427246094,
    'tcc': 0.6294848918914795,
    't_1000': 288.50189208984375,
    't_925': 284.2601318359375,
    't_850': 281.25579833984375,
    't_700': 273.9064636230469,
    'u_1000': -0.3982798755168915,
    'u_925': 0.2242080569267273,
    'u_850': 1.0792275667190552,
    'u_700': 3.2283222675323486,
    'v_1000': 0.1467936784029007,
    'v_925': 0.1286812722682953,
    'v_850': 0.03933039307594299,
    'v_700': -0.02824198640882969,
    'z_1000': 942.7501831054688,
    'z_925': 7371.5751953125,
    'z_850': 14263.0771484375,
    'z_700': 29794.1171875,
    'q_1000': 0.009408126585185528,
    'q_925': 0.008069725707173347,
    'q_850': 0.00611522002145648,
    'q_700': 0.0032571181654930115
}
stds_dict = {
    '2t': np.float64(15.416738534020974),
    'msl': np.float64(1155.7307212322428),
    '10u': np.float64(5.568044406702828),
    '10v': np.float64(4.567121908139054),
    'tp': np.float64(0.002447679867059621),
    'sst': np.float64(10.330162505268438),
    'tcwv': np.float64(17.24232112969287),
    'tcc': np.float64(0.36417275425138707),
    't_1000': np.float64(13.3624105652117),
    't_925': np.float64(12.732162863842351),
    't_850': np.float64(12.366676074193236),
    't_700': np.float64(11.621001181354009),
    'u_1000': np.float64(6.230015657506282),
    'u_925': np.float64(8.052301151410559),
    'u_850': np.float64(8.340163470891408),
    'u_700': np.float64(9.354396712455802),
    'v_1000': np.float64(5.151321737803057),
    'v_925': np.float64(6.186798980766869),
    'v_850': np.float64(5.871119739629767),
    'v_700': np.float64(6.425596026965733),
    'z_1000': np.float64(936.1515302022424),
    'z_925': np.float64(1059.7839992187087),
    'z_850': np.float64(1253.2425443624231),
    'z_700': np.float64(1801.0211686707073),
    'q_1000': np.float64(0.005841925212812825),
    'q_925': np.float64(0.005067701844307394),
    'q_850': np.float64(0.004285320445588393),
    'q_700': np.float64(0.0028702109939330775)
}

In [143]:
print("Создание даталоадера")
loader = WeatherAutoencoderLoader(bgen, means_dict, stds_dict, device=device)

Создание даталоадера


In [144]:
print("Получение тестового батча")
X_batch, y_batch = next(iter(loader))

print("Успешно загружено!")
print("Форма входного тензора X:", X_batch.shape)
print("Форма целевого тензора y:", y_batch.shape)

Получение тестового батча
Успешно загружено!
Форма входного тензора X: torch.Size([8, 28, 128, 128])
Форма целевого тензора y: torch.Size([8, 28, 128, 128])


## Модели

In [146]:
import torch
from torch import nn

class SafeQuantizeSTE(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, num_bits=4):
        # защита от NaN/Inf
        x = torch.nan_to_num(x, nan=0.0, posinf=1.0, neginf=-1.0)

        # динамическое масштабирование
        min_val = x.min()
        max_val = x.max()
        scale = torch.clamp(max_val - min_val, min=1e-7)

        x_normalized = (x - min_val) / scale

        levels = (2 ** num_bits) - 1
        x_scaled = x_normalized * levels
        x_rounded = torch.round(x_scaled)

        # в исходный масштаб
        x_quantized = (x_rounded / levels) * scale + min_val

        return x_quantized

    @staticmethod
    def backward(ctx, grad_output):
        clean_grad = torch.nan_to_num(grad_output, nan=0.0, posinf=0.0, neginf=0.0)

        return clean_grad, None

def quantize_ste(x, num_bits=4):
    return SafeQuantizeSTE.apply(x, num_bits)

In [147]:
class EncoderWithQuantization(nn.Module):
    def __init__(self, in_channels=28, latent_channels=14, num_levels=15):
        super().__init__()
        self.num_levels = num_levels

        self.encoder_net = nn.Sequential(
            nn.Conv2d(in_channels, 20, kernel_size=5, stride=2, padding=2),
            nn.BatchNorm2d(20),
            nn.ReLU(),
            nn.Conv2d(20, latent_channels, kernel_size=5, stride=2, padding=2),
            nn.BatchNorm2d(latent_channels),
            nn.Tanh()
        )

    def forward(self, x):
        x = self.encoder_net(x)
        
        # Масштабируем интервал [-1, 1] в диапазон целых чисел
        x_scaled = (x + 1) * (self.num_levels / 2)
        
        # Квантование с помощью STE
        x_quantized = quantize_ste(x_scaled)
        
        # (Опционально) Возвращаем к исходному масштабу для декодера
        x_normalized = (x_quantized / (self.num_levels / 2)) - 1
        return x_normalized

In [148]:
import torch
import torch.nn as nn
import struct
import torchac
from typing import Tuple


class Decoder(nn.Module):
    def __init__(self, latent_channels=14, out_channels=28):
        super().__init__()
        self.deconv1 = nn.ConvTranspose2d(latent_channels, 20, kernel_size=5, stride=2, padding=2, output_padding=1)
        self.bnorm = nn.BatchNorm2d(20)
        self.deconv2 = nn.ConvTranspose2d(20, out_channels, kernel_size=5, stride=2, padding=2, output_padding=1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, z):
        z = self.bnorm(self.deconv1(z))
        z = torch.relu(z)
        z = self.deconv2(z)
        # z = self.sigmoid(z)
        return z

# ---------- вспомогательные функции для энтропийного кодирования ----------
def encode_latents_with_scale(latents_int: torch.Tensor, scale: float) -> bytes:
    min_val, max_val = -128, 127

    latents_cpu = latents_int.detach().cpu()

    symbols = (latents_cpu.flatten().to(torch.int32) - min_val).clamp(0, max_val - min_val)
    symbols_batch = symbols.unsqueeze(0).to(torch.int16)
    num_symbols = symbols_batch.shape[1]

    alphabet = torch.arange(min_val, max_val + 1, dtype=torch.float32)
    probs = 0.5 * torch.exp(-torch.abs(alphabet) / scale) + 1e-10
    probs = probs / probs.sum()
    cdf_single = torch.cat([torch.tensor([0.0]), torch.cumsum(probs, dim=0)]).unsqueeze(0)

    cdf_batch = cdf_single.unsqueeze(1).expand(1, num_symbols, -1)

    encoded = torchac.encode_float_cdf(
        cdf_batch,
        symbols_batch,
        check_input_bounds=True
    )
    return encoded

def decode_latents_with_scale(encoded_bytes: bytes, shape: Tuple[int, int, int], scale: float) -> torch.Tensor:
    """Декодирование с исправленными размерностями CDF"""
    min_val, max_val = -128, 127
    num_symbols = shape[0] * shape[1] * shape[2]

    alphabet = torch.arange(min_val, max_val + 1, dtype=torch.float32)
    probs = 0.5 * torch.exp(-torch.abs(alphabet) / scale) + 1e-10
    probs = probs / probs.sum()
    cdf_single = torch.cat([torch.tensor([0.0]), torch.cumsum(probs, dim=0)]).unsqueeze(0)

    cdf_batch = cdf_single.unsqueeze(1).expand(1, num_symbols, -1)

    decoded_symbols = torchac.decode_float_cdf(cdf_batch, encoded_bytes)
    latents_flat = decoded_symbols + min_val
    return latents_flat.reshape(shape).to(torch.int32)

def compute_B(encoder: EncoderWithQuantization, decoder: Decoder, image: torch.Tensor, scale: float = 1.0) -> int:
    """
    Принимает модель и изображение (1, 28, H, W).
    Возвращает размер B в байтах.
    """
    with torch.no_grad():
        latents_float = encoder(image)
        latents_int = latents_float.to(torch.int32)

        C, H, W = latents_int.shape[1], latents_int.shape[2], latents_int.shape[3]

        header = bytearray()
        header.append(1)
        header.extend(struct.pack('>HHH', C, H, W))
        header.extend(struct.pack('>f', scale))

        encoded_latents = encode_latents_with_scale(latents_int, scale)

        bitstream = bytes(header) + encoded_latents
        return len(bitstream)

In [149]:
import torch
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

encoder = EncoderWithQuantization(latent_channels=14).to(device)
decoder = Decoder(latent_channels=14).to(device)

In [150]:
optimizer = optim.Adam(
    list(encoder.parameters()) + list(decoder.parameters()),
    lr=1e-3
)

criterion = nn.SmoothL1Loss(beta=1.0) 

In [ ]:
encoder.train()
decoder.train()

print(f"Запуск тестового цикла на устройстве: {device}")
losses = []
i = 0

for X_batch, _ in tqdm(loader, total=len(loader), desc="Testing Autoencoder"):
    optimizer.zero_grad()

    latent_features = encoder(X_batch)
    reconstructed_X = decoder(latent_features)

    loss = criterion(reconstructed_X, X_batch)

    losses.append(loss.item())
    loss.backward()

    # клиппинг градиентов
    torch.nn.utils.clip_grad_norm_(encoder.parameters(), max_norm=1.0)
    torch.nn.utils.clip_grad_norm_(decoder.parameters(), max_norm=1.0)

    optimizer.step()

    if i % 10 == 0:
        print(" -", sum(losses) / len(losses))

    i += 1

print(f"Тест успешно завершен! Финальный Loss: {loss.item():.4f}")


In [138]:
torch.save(encoder.state_dict(), "/kaggle/working/encoder.pt")
torch.save(decoder.state_dict(), "/kaggle/working/decoder.pt")